In [1]:
import pandas as pd
import numpy as np


In [3]:
url="https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
data=pd.read_csv(url)
df=pd.DataFrame(data)
df.head()


,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [10]:
df.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [7]:
df['horsepower'].median()

np.float64(254.0)

In [13]:
df["horsepower"] = df["horsepower"].fillna(df["horsepower"].mean())
df["acceleration"] = df["acceleration"].fillna(df["acceleration"].mean())

In [14]:
df.isnull().sum()


model_year             0
origin                 0
fuel_type              0
drivetrain             0
num_doors              0
engine_displacement    0
num_cylinders          0
horsepower             0
vehicle_weight         0
acceleration           0
fuel_efficiency_mpg    0
dtype: int64

In [22]:
y = df["fuel_efficiency_mpg"]
  

In [21]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

df_train, df_val = train_test_split(
    df_full_train,
    test_size=0.25,
    random_state=42
)

In [23]:
len(df_train), len(df_val), len(df_test)

(6000, 2000, 2000)

In [26]:
target = "fuel_efficiency_mpg"

X_train = df_train.drop(columns=[target])
y_train = df_train[target]

X_val = df_val.drop(columns=[target])
y_val = df_val[target]

X_test = df_test.drop(columns=[target])
y_test = df_test[target]

In [27]:
X_train.shape, y_train.shape

((6000, 10), (6000,))

In [28]:
X_train.dtypes

model_year               int64
origin                     str
fuel_type                  str
drivetrain                 str
num_doors                int64
engine_displacement      int64
num_cylinders            int64
horsepower             float64
vehicle_weight           int64
acceleration           float64
dtype: object

In [29]:
mean_horsepower = X_train["horsepower"].mean()

mean_horsepower

np.float64(254.07391839307243)

In [30]:
X_train = X_train.copy()

X_train["horsepower"] = X_train["horsepower"].fillna(mean_horsepower)

In [31]:
X_train["horsepower"].isna().sum()

np.int64(0)

In [32]:
X_train[["origin", "fuel_type", "drivetrain"]].nunique()

origin        3
fuel_type     3
drivetrain    3
dtype: int64

In [33]:
for col in ["origin", "fuel_type", "drivetrain"]:
    print(col)
    print(X_train[col].unique())
    print()

origin
<ArrowStringArray>
['Asia', 'Europe', 'USA']
Length: 3, dtype: str

fuel_type
<ArrowStringArray>
['Gasoline', 'Diesel', 'Hybrid']
Length: 3, dtype: str

drivetrain
<ArrowStringArray>
['Front-wheel drive', 'Rear-wheel drive', 'All-wheel drive']
Length: 3, dtype: str



In [34]:
X_train = pd.get_dummies(
    X_train,
    columns=["origin", "fuel_type", "drivetrain"],
    dtype=int
)

In [35]:
X_train.shape

(6000, 16)

In [36]:
X_val = X_val.copy()

X_val["horsepower"] = X_val["horsepower"].fillna(mean_horsepower)

In [37]:
X_val = pd.get_dummies(
    X_val,
    columns=["origin", "fuel_type", "drivetrain"],
    dtype=int
)

In [38]:
X_val.shape

(2000, 16)

In [39]:
X_val.isna().sum().sum()

np.int64(0)

In [54]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](16,)","[ 0.1 ,-0.02,-0. ,...,-0.53, 0.69,-0.16]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](16,)","['model_year','num_doors','engine_displacement',..., 'drivetrain_All-wheel drive','drivetrain_Front-wheel drive', 'drivetrain_Rear-wheel drive']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,-155.6
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,16
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(13)


In [50]:
model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](16,)","[ 0.1 ,-0.02,-0. ,...,-0.53, 0.69,-0.16]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](16,)","['model_year','num_doors','engine_displacement',..., 'drivetrain_All-wheel drive','drivetrain_Front-wheel drive', 'drivetrain_Rear-wheel drive']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,-155.6
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,16
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(13)


In [55]:
y_pred = model.predict(X_val)

In [56]:
from sklearn.metrics import root_mean_squared_error

rmse = root_mean_squared_error(y_val, y_pred)

rmse

1.5918413342216253

In [57]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=0)

ridge.fit(X_train, y_train)

y_pred = ridge.predict(X_val)

root_mean_squared_error(y_val, y_pred)

1.592041790974403

In [58]:
alphas = [0, 0.01, 0.1, 1, 5, 10, 100]

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train, y_train)

    y_pred = ridge.predict(X_val)

    rmse = root_mean_squared_error(y_val, y_pred)

    print(alpha, rmse)

0 1.592041790974403
0.01 1.591841200536654
0.1 1.591840001855551
1 1.591828457982189
5 1.5917867930399876
10 1.591756388275806
100 1.5945846645340875


In [61]:
alpha = 10
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [62]:
for train_idx, val_idx in kf.split(X_train):
    print(len(train_idx), len(val_idx))

4800 1200
4800 1200
4800 1200
4800 1200
4800 1200


In [63]:
rmses = []

for train_idx, val_idx in kf.split(X_train):
    X_train_fold = X_train.iloc[train_idx]
    X_val_fold = X_train.iloc[val_idx]

    y_train_fold = y_train.iloc[train_idx]
    y_val_fold = y_train.iloc[val_idx]

    model = Ridge(alpha=10)
    model.fit(X_train_fold, y_train_fold)

    y_pred = model.predict(X_val_fold)

    rmse = root_mean_squared_error(y_val_fold, y_pred)

    rmses.append(rmse)

    print(rmse)

1.6057108814329264
1.6431026641834652
1.6304744701333849
1.6588210535709642
1.6463042514169803


In [64]:
import numpy as np

np.std(rmses)

np.float64(0.018008290456045725)

In [66]:
CAR_COLUMNS = [
    "engine_displacement",
    "horsepower",
    "vehicle_weight",
    "model_year"
]

In [67]:
data = df[CAR_COLUMNS + ["fuel_efficiency_mpg"]]

seed_scores = []

for seed in range(10):
    indices = np.arange(len(data))
    np.random.RandomState(seed).shuffle(indices)

    n_train = int(len(data) * 0.6)
    n_validation = int(len(data) * 0.2)

    train = data.iloc[indices[:n_train]]
    validation = data.iloc[
        indices[n_train:n_train + n_validation]
    ]

    X_train_seed = train[CAR_COLUMNS].fillna(0)
    y_train_seed = train["fuel_efficiency_mpg"]

    X_val_seed = validation[CAR_COLUMNS].fillna(0)
    y_val_seed = validation["fuel_efficiency_mpg"]

    model = LinearRegression()
    model.fit(X_train_seed, y_train_seed)

    y_pred = model.predict(X_val_seed)

    rmse = root_mean_squared_error(y_val_seed, y_pred)

    seed_scores.append(rmse)

    print(seed, rmse)

0 2.235933257930576
1 2.1962476943098297
2 2.158729740635674
3 2.203904975216485
4 2.1986707448682057
5 2.241032859090124
6 2.2642187658499355
7 2.185597205149688
8 2.217510003542266
9 2.2040142196450327


In [69]:
np.std(seed_scores)

np.float64(0.028710287909102388)

In [70]:
X_test_q6 = df_test[CAR_COLUMNS].fillna(0)
y_test_q6 = df_test["fuel_efficiency_mpg"]

X_train_q6 = df_train[CAR_COLUMNS].fillna(0)
y_train_q6 = df_train["fuel_efficiency_mpg"]

In [71]:
model = LinearRegression()
model.fit(X_train_q6, y_train_q6)

y_pred_test = model.predict(X_test_q6)

rmse_test = root_mean_squared_error(y_test_q6, y_pred_test)

print(rmse_test)

2.216874472281536


In [75]:
data = df[CAR_COLUMNS + ["fuel_efficiency_mpg"]]

indices = np.arange(len(data))
np.random.RandomState(9).shuffle(indices)

n_train = int(len(data) * 0.6)
n_validation = int(len(data) * 0.2)

train_9 = data.iloc[indices[:n_train]]
validation_9 = data.iloc[indices[n_train:n_train + n_validation]]
test_9 = data.iloc[indices[n_train + n_validation:]]

In [76]:
print(train_9.shape)
print(validation_9.shape)
print(test_9.shape)

(6000, 5)
(2000, 5)
(2000, 5)


In [84]:
combined = pd.concat([train_9, validation_9])

X_train_q6 = combined[CAR_COLUMNS].fillna(0)
y_train_q6 = combined["fuel_efficiency_mpg"]

X_test_q6 = test_9[CAR_COLUMNS].fillna(0)
y_test_q6 = test_9["fuel_efficiency_mpg"]

In [86]:
combined = pd.concat([train_9, validation_9])

X_train_q6 = combined[CAR_COLUMNS].fillna(0)
y_train_q6 = combined["fuel_efficiency_mpg"]

X_test_q6 = test_9[CAR_COLUMNS].fillna(0)
y_test_q6 = test_9["fuel_efficiency_mpg"]

In [87]:
from sklearn.linear_model import Ridge

model = Ridge(alpha=0.001)

model.fit(X_train_q6, y_train_q6)

,"alpha alpha: float or array-like of shape (n_targets,), default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.See :ref:`sphx_glr_auto_examples_linear_model_plot_ridge_coeffs.py`for an illustration of the effect of alpha on the model coefficients.",0.001
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga

In [89]:
y_pred_test = model.predict(X_test_q6)

rmse_test = root_mean_squared_error(
    y_test_q6,
    y_pred_test
)

print(rmse_test)

2.2299014772213828


In [90]:
Ridge(alpha=0.001)

,"alpha alpha: float or array-like of shape (n_targets,), default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.See :ref:`sphx_glr_auto_examples_linear_model_plot_ridge_coeffs.py`for an illustration of the effect of alpha on the model coefficients.",0.001
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga

In [91]:
def train_linear_regression_reg(X, y, r=0.0):
    X = np.column_stack([np.ones(len(X)), X])

    XTX = X.T @ X

    reg = r * np.eye(XTX.shape[0])
    reg[0, 0] = 0

    XTX_reg = XTX + reg

    w = np.linalg.solve(XTX_reg, X.T @ y)

    return w

In [92]:
X_train_q6 = combined[CAR_COLUMNS].fillna(0)
y_train_q6 = combined["fuel_efficiency_mpg"]

X_test_q6 = test_9[CAR_COLUMNS].fillna(0)
y_test_q6 = test_9["fuel_efficiency_mpg"]

w = train_linear_regression_reg(
    X_train_q6,
    y_train_q6,
    r=0.001
)

In [93]:
X_test_matrix = np.column_stack([
    np.ones(len(X_test_q6)),
    X_test_q6
])

y_pred_test = X_test_matrix @ w

rmse_test = root_mean_squared_error(
    y_test_q6,
    y_pred_test
)

print(rmse_test)

2.229901477221545


In [94]:
print(test_9.index[:10])
print(test_9.index[-10:])

Index([5463, 9878, 8503, 2571, 6851, 3197, 3088, 7627, 2025, 7816], dtype='int64')
Index([4532, 4673, 9341, 5014, 9979, 6200, 501, 6782, 4444, 8574], dtype='int64')


In [95]:
idx = np.arange(len(data))

rng = np.random.default_rng(9)
rng.shuffle(idx)

n_train = int(len(data) * 0.6)
n_val = int(len(data) * 0.2)

train_9 = data.iloc[idx[:n_train]]
validation_9 = data.iloc[idx[n_train:n_train + n_val]]
test_9 = data.iloc[idx[n_train + n_val:]]

print(train_9.shape)
print(validation_9.shape)
print(test_9.shape)
print(test_9.index[:10])

(6000, 5)
(2000, 5)
(2000, 5)
Index([6709, 412, 1011, 8214, 6373, 8233, 8554, 895, 5385, 1305], dtype='int64')


In [97]:
print(df.index[:10])
print(df.index[-10:])
print(df.shape)

RangeIndex(start=0, stop=10, step=1)
RangeIndex(start=9990, stop=10000, step=1)
(10000, 11)


In [99]:
X_train_q3 = df_train[CAR_COLUMNS].fillna(0)
y_train_q3 = df_train["fuel_efficiency_mpg"]

X_val_q3 = df_val[CAR_COLUMNS].fillna(0)
y_val_q3 = df_val["fuel_efficiency_mpg"]

model_0 = LinearRegression()
model_0.fit(X_train_q3, y_train_q3)

y_pred_0 = model_0.predict(X_val_q3)

rmse_0 = root_mean_squared_error(y_val_q3, y_pred_0)

print("RMSE with 0:", rmse_0)

RMSE with 0: 2.180045839487052


In [100]:
mean_horsepower = df_train["horsepower"].mean()

X_train_mean = df_train[CAR_COLUMNS].copy()
X_val_mean = df_val[CAR_COLUMNS].copy()

X_train_mean["horsepower"] = X_train_mean["horsepower"].fillna(mean_horsepower)
X_val_mean["horsepower"] = X_val_mean["horsepower"].fillna(mean_horsepower)

model_mean = LinearRegression()
model_mean.fit(X_train_mean, y_train_q3)

y_pred_mean = model_mean.predict(X_val_mean)

rmse_mean = root_mean_squared_error(y_val_q3, y_pred_mean)

print("Mean horsepower:", mean_horsepower)
print("RMSE with mean:", rmse_mean)

Mean horsepower: 254.07391839307243
RMSE with mean: 2.180045839487052
